# Módulo 06 · Notebook 2: *Baseline* y primeros modelos para House Prices

**Pregunta de negocio (módulo 03):** ¿podemos dar un avalúo automático con un error típico (MAPE) **menor al 15 %**?

**Plan**
1. Dividir los datos en **train / validación / test**.
2. Construir un ***baseline*** (el modelo "tonto" que hay que superar).
3. Probar modelos de complejidad creciente y vigilar el *overfitting*.
4. Validación cruzada, importancia de variables y análisis de errores.
5. Evaluar **una sola vez** en test.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

from intro_analitica import config
from intro_analitica.datos import cargar_house_prices_limpio
from intro_analitica.metricas import brecha_overfitting, metricas_regresion, tabla_comparativa

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)

df = cargar_house_prices_limpio()
print(df.shape)

## 1. Variables del modelo

Usamos las variables que el EDA (módulo 05) señaló como importantes. Están definidas en `config.py` porque **la API del módulo 08 pedirá exactamente estas mismas**.

In [ ]:
NUMERICAS = config.FEATURES_NUMERICAS
CATEGORICAS = config.FEATURES_CATEGORICAS
print("Numéricas  :", NUMERICAS)
print("Categóricas:", CATEGORICAS)

X = df[NUMERICAS + CATEGORICAS]
y = df[config.TARGET]

## 2. Train / validación / test

```
 ┌──────────────────────── 100 % de los datos ───────────────────────┐
 │        TRAIN (60 %)            │  VALIDACIÓN (20 %) │  TEST (20 %) │
 │  el modelo aprende aquí        │  elegimos modelo   │  se usa   │
 │                                │  e hiperparámetros │  UNA vez     │
 └────────────────────────────────┴────────────────────┴──────────────┘
```

**¿Por qué tres conjuntos?** Si eliges el modelo mirando el test, el test deja de ser "datos nuevos": terminas ajustándote a él sin darte cuenta.

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=config.SEMILLA)
X_train, X_valid, y_train, y_valid = train_test_split(X_temp, y_temp, test_size=0.25, random_state=config.SEMILLA)
print(f"Train: {len(X_train)} · Validación: {len(X_valid)} · Test: {len(X_test)}")

### Preprocesamiento manual (y por qué el módulo 07 existe)

Los modelos necesitan números, así que codificamos las categóricas con one-hot. **Ajustamos el encoder solo con train** para evitar *data leakage*.
`handle_unknown="ignore"` evita un error si en validación o test aparece una categoría que no estaba en train.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(X_train[CATEGORICAS])


def preparar(X_parte: pd.DataFrame) -> pd.DataFrame:
    """Numéricas tal cual + categóricas en one-hot (con el encoder ajustado en train)."""
    dummies = pd.DataFrame(
        encoder.transform(X_parte[CATEGORICAS]),
        columns=encoder.get_feature_names_out(CATEGORICAS),
        index=X_parte.index,
    )
    return pd.concat([X_parte[NUMERICAS], dummies], axis=1)


Xtr, Xva, Xte = preparar(X_train), preparar(X_valid), preparar(X_test)
print("Columnas después del one-hot:", Xtr.shape[1])

> Tener que acordarse de `fit` en train, `transform` en todo, guardar el encoder… es fácil equivocarse. En el **módulo 07** un `Pipeline` hará todo esto por nosotros.

---
## 3. El *baseline*: el modelo a superar

El *baseline* más simple: **predecir siempre la mediana** del train. Si un modelo "inteligente" no le gana a esto, no sirve.

In [ ]:
resultados = {}  # {nombre: métricas en validación}
entrenamiento = {}  # {nombre: métricas en train}


def evaluar(nombre, modelo, X_tr=Xtr, X_va=Xva, y_tr=y_train, transformar_y=None):
    """Entrena, calcula métricas en train y validación y las guarda."""
    if transformar_y == "log":
        modelo.fit(X_tr, np.log1p(y_tr))
        pred_tr, pred_va = np.expm1(modelo.predict(X_tr)), np.expm1(modelo.predict(X_va))
    else:
        modelo.fit(X_tr, y_tr)
        pred_tr, pred_va = modelo.predict(X_tr), modelo.predict(X_va)
    entrenamiento[nombre] = metricas_regresion(y_tr, pred_tr)
    resultados[nombre] = metricas_regresion(y_valid, pred_va)
    return modelo


evaluar("0. Baseline (mediana)", DummyRegressor(strategy="median"))
pd.DataFrame(resultados).T

El *baseline* se equivoca en promedio un **~31 %** (MAPE). Ese es el número a vencer.

---
## 4. Modelos de complejidad creciente

### 4.1 Regresión lineal

In [ ]:
evaluar("1. Lineal (solo numéricas)", LinearRegression(), X_tr=Xtr[NUMERICAS], X_va=Xva[NUMERICAS])
lineal = evaluar("2. Lineal (+ categóricas)", LinearRegression())
evaluar("3. Lineal + log(target)", LinearRegression(), transformar_y="log")
tabla_comparativa(resultados)

**Lectura:**
- La regresión lineal ya deja el MAPE en **menos de la mitad** que el *baseline* (~12 % frente a ~31 %).
- Agregar las categóricas (barrio, calidad de la cocina…) ayuda.
- Usar **log(precio)** (hallazgo #1 del EDA) mejora aún más: el modelo deja de "pelear" con la cola de casas caras.

### 4.2 Árbol de decisión y el *overfitting* en vivo

In [ ]:
profundidades = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20, None]
curva = []
for prof in profundidades:
    arbol = DecisionTreeRegressor(max_depth=prof, random_state=config.SEMILLA).fit(Xtr, y_train)
    curva.append(
        {
            "max_depth": str(prof),
            "MAE_train": metricas_regresion(y_train, arbol.predict(Xtr))["MAE"],
            "MAE_valid": metricas_regresion(y_valid, arbol.predict(Xva))["MAE"],
        }
    )
curva = pd.DataFrame(curva).set_index("max_depth")

ax = curva.plot(marker="o", figsize=(9, 4))
ax.set(title="Árbol de decisión: error vs. profundidad", ylabel="MAE (USD)")
plt.show()
curva.T

Con `max_depth=None` el árbol **memoriza** el train (MAE de train ≈ 0), pero en validación se equivoca en promedio por casi USD 30.000: ***overfitting*** de libro.
La mejor profundidad está en el "valle" de la curva de validación.

In [ ]:
mejor_prof = int(curva["MAE_valid"].drop("None").idxmin())
print("Mejor profundidad en validación:", mejor_prof)
evaluar(f"4. Árbol (max_depth={mejor_prof})", DecisionTreeRegressor(max_depth=mejor_prof, random_state=config.SEMILLA))

### 4.3 *Ensembles*: muchos árboles juntos

| Modelo | Idea |
|---|---|
| **Random Forest** | Cientos de árboles entrenados con muestras distintas; se **promedian** sus predicciones (reduce la varianza) |
| **Gradient Boosting** | Árboles en **secuencia**: cada uno corrige los errores del anterior (reduce el sesgo) |

In [ ]:
bosque = evaluar("5. Random Forest", RandomForestRegressor(n_estimators=300, random_state=config.SEMILLA, n_jobs=-1))
evaluar("6. Gradient Boosting", HistGradientBoostingRegressor(random_state=config.SEMILLA))
evaluar("7. Gradient Boosting + log", HistGradientBoostingRegressor(random_state=config.SEMILLA), transformar_y="log")

**XGBoost (opcional):** es la versión más popular de *gradient boosting* (la usó el profesor en su notebook). Viene en el `pyproject.toml` del curso.

In [ ]:
try:
    from xgboost import XGBRegressor

    evaluar(
        "8. XGBoost + log",
        XGBRegressor(n_estimators=1000, learning_rate=0.03, max_depth=4, subsample=0.8, random_state=config.SEMILLA),
        transformar_y="log",
    )
except ImportError:
    print("XGBoost no está instalado (uv sync lo instala). Seguimos sin él.")

---
## 5. La tabla comparativa

In [ ]:
comparacion = tabla_comparativa(resultados)
comparacion["MAE_train"] = [entrenamiento[m]["MAE"] for m in comparacion.index]
comparacion["brecha_%"] = [brecha_overfitting(entrenamiento[m]["MAE"], resultados[m]["MAE"]) for m in comparacion.index]
comparacion

**Cómo leer la tabla:**
- **MAPE**: ¿cumplimos el criterio de negocio (< 15 %)?
- **brecha_%**: cuánto peor le va en validación que en train. Los *ensembles* (Random Forest, Gradient Boosting) tienen brechas enormes: casi memorizan el train. Aun así, su error en validación es de los mejores. **Lo que importa es el error en datos nuevos**; la brecha es una alerta para regularizar.
- Los *ensembles* y la **lineal con log** quedan muy cerca: todos alrededor del 10-11 % de MAPE, **muy por debajo del criterio de 15 %**.

> **Desempeño vs. interpretabilidad:** si la regresión lineal está muy cerca del mejor modelo, puede valer la pena elegirla porque **se explica** fácilmente ("cada m² suma X dólares").

In [ ]:
ax = comparacion["MAPE_%"].sort_values().plot(kind="barh", figsize=(8, 4), color="teal")
ax.axvline(15, color="red", ls="--", label="Criterio de negocio (15 %)")
ax.set(title="MAPE en validación (menor es mejor)", xlabel="MAPE %")
ax.legend()
plt.show()

---
## 6. Validación cruzada: ¿y si tuvimos suerte con el split?

Un solo split de validación puede salir "fácil" o "difícil" por azar. La **validación cruzada (k-fold)** entrena y evalúa k veces, con un pedazo distinto como validación cada vez.

```
 Fold 1: [VAL][ tr ][ tr ][ tr ][ tr ]
 Fold 2: [ tr ][VAL][ tr ][ tr ][ tr ]
 ...                                       → promedio ± desviación
 Fold 5: [ tr ][ tr ][ tr ][ tr ][VAL]
```

In [ ]:
X_cv, y_cv = pd.concat([Xtr, Xva]), pd.concat([y_train, y_valid])
kfold = KFold(n_splits=5, shuffle=True, random_state=config.SEMILLA)

for nombre, modelo in {
    "Lineal (+ categóricas)": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=config.SEMILLA, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingRegressor(random_state=config.SEMILLA),
}.items():
    mape = -cross_val_score(modelo, X_cv, y_cv, cv=kfold, scoring="neg_mean_absolute_percentage_error") * 100
    print(f"{nombre:<24} MAPE = {mape.mean():5.2f} % ± {mape.std():.2f}   (folds: {np.round(mape, 1)})")

> Aquí aplicamos el encoder **antes** de la validación cruzada (ajustado solo con train). Lo correcto es reajustarlo dentro de cada fold… otra razón para usar **Pipelines** (módulo 07).

---
## 7. ¿Qué aprendió el modelo? Importancia de variables

In [ ]:
importancias = pd.Series(bosque.feature_importances_, index=Xtr.columns).sort_values(ascending=False)
ax = importancias.head(12).sort_values().plot(kind="barh", figsize=(7, 5), color="slateblue")
ax.set(title="Random Forest: las 12 variables más importantes")
plt.show()

Coincide con el EDA: **calidad general**, **área habitable** y **área del sótano** dominan. Cuando el modelo y el EDA cuentan la misma historia, ganamos confianza.

---
## 8. Análisis de errores: ¿en qué casas se equivoca más?

In [ ]:
pred_va = np.expm1(LinearRegression().fit(Xtr, np.log1p(y_train)).predict(Xva))
errores = X_valid.assign(real=y_valid, predicho=pred_va)
errores["error_%"] = (errores["predicho"] / errores["real"] - 1) * 100
errores["rango_precio"] = pd.qcut(errores["real"], 4, labels=["barata", "media-baja", "media-alta", "cara"])

print(errores.groupby("rango_precio", observed=True)["error_%"].apply(lambda e: e.abs().mean()).round(1).to_frame("MAPE_%"))
errores.reindex(errores["error_%"].abs().sort_values(ascending=False).index).head(5)[
    ["Neighborhood", "OverallQual", "GrLivArea", "real", "predicho", "error_%"]
]

Mirar los peores errores uno por uno suele revelar algo: casas atípicas, ventas especiales (`SaleCondition`) o datos raros. Es el insumo para la **siguiente iteración** de CRISP-DM.

---
## 9. Evaluación final en TEST

Elegimos el modelo **con validación** y lo evaluamos **una sola vez** en test. Tomamos *Gradient Boosting + log*: está entre los mejores MAPE de validación y fue el mejor en la validación cruzada.
(La lineal + log queda a menos de 1 punto: sería la elección si el negocio necesita **explicar** cada predicción.)

Lo reentrenamos con train + validación (más datos = mejor modelo).

In [ ]:
final = HistGradientBoostingRegressor(random_state=config.SEMILLA)
final.fit(X_cv, np.log1p(y_cv))
pred_test = np.expm1(final.predict(Xte))
metricas_test = metricas_regresion(y_test, pred_test)
metricas_test

In [ ]:
dentro_15 = (np.abs(pred_test / y_test - 1) <= 0.15).mean()
print(f"MAPE en test: {metricas_test['MAPE_%']} %  → criterio < 15 %: {'CUMPLE' if metricas_test['MAPE_%'] < 15 else 'NO CUMPLE'}")
print(f"Casas con error ≤ 15 %: {dentro_15:.0%}  (criterio de negocio: 80 %)  → {'cumple' if dentro_15 >= 0.8 else 'no cumple'}")

**Conclusión para el negocio:** un modelo con 14 variables estima el precio con un error típico de alrededor del 10 %, frente al 31 % de no tener modelo.
Hay que comunicar también **sus límites**: datos de 2006-2010, casas atípicas y barrios con pocas ventas.

---
## Ejercicios guiados (20 min)

1. **Más variables:** agrega `TotalSF`, `TotalBath` y `HouseAge` (del módulo 04) a las numéricas. ¿Mejora la regresión lineal con log?
2. **Sin los outliers:** elimina del **train** las casas con `GrLivArea > 4000` y reentrena la lineal con log. ¿Cambia el MAPE de validación?
3. **Random Forest regularizado:** prueba `max_depth` 8, 12 y 16 y `min_samples_leaf` 1, 3 y 5. ¿Puedes reducir la brecha sin empeorar la validación?
4. **Reflexión:** ¿por qué no deberíamos volver a ajustar el modelo después de ver el resultado en test?

Soluciones en `ejercicios/soluciones/solucion_06_modelos.py`.

In [ ]:
# Tu código aquí